# 1.4 Reddit — Sentiment Analysis

Four sentiment methods applied to Reddit post text, in order of complexity:

| # | Method | Type | Output |
|---|---|---|---|
| 1 | **VADER** | Rule-based lexicon | `vader_compound` ∈ [−1, +1] |
| 2 | **Twitter-RoBERTa** | Pre-trained transformer (fine-tuned on tweets) | `roberta_sentiment` = pos − neg ∈ [−1, +1] |
| 3 | **TextBlob** | Pattern lexicon | `tb_polarity` ∈ [−1, +1], `tb_subjectivity` ∈ [0, 1] |
| 4 | **NRCLex** | Emotion lexicon | 8 emotion dimensions |

**Predictive features saved:** Twitter-RoBERTa + NRCLex.

**Input:** `Data/2_Silver/Reddit/reddit_posts_clean.parquet`  
**Output:** `Data/2_Silver/Reddit/sentiment_reddit.csv`

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.lines as mlines

sys.path.insert(0, os.path.abspath('../..'))
from house_style import (
    apply_style, REPUBLICAN, DEMOCRAT, NEUTRAL, ACCENT,
    TEXT_PRIMARY, TEXT_MUTED, BG_DARK, BG_PANEL,
    EVENTS, add_events
)
apply_style()

import nltk
nltk.download('stopwords',                    quiet=True)
nltk.download('punkt',                        quiet=True)
nltk.download('punkt_tab',                    quiet=True)
nltk.download('averaged_perceptron_tagger',   quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from nrclex import NRCLex
from textblob import TextBlob

DATA_PATH = '../../Data/2_Silver/Reddit/reddit_posts_clean.parquet'
OUT_PATH  = '../../Data/2_Silver/Reddit/sentiment_reddit.csv'

SUBREDDIT_COLORS = {
    'conservative': REPUBLICAN,
    'trump':        '#c0392b',
    'republican':   '#e07b39',
    'democrats':    DEMOCRAT,
    'liberal':      '#5dade2',
    'worldnews':    '#2ecc71',
    'politics':     '#f39c12',
}

BUZZ_COLORS = {'TrumpBuzz': REPUBLICAN, 'HarrisBuzz': DEMOCRAT, 'ElectionBuzz': NEUTRAL}
BUZZ_ORDER  = ['TrumpBuzz', 'HarrisBuzz', 'ElectionBuzz']
POL_COLORS  = {'Positive': '#2ecc71', 'Neutral': TEXT_MUTED, 'Negative': REPUBLICAN}
PURPLE      = '#9b5de5'

EMOTION_COLS = ['fear', 'anger', 'anticipation', 'trust', 'surprise', 'sadness', 'disgust', 'joy']

def event_legend(fig, ax, loc='upper left'):
    data_h = ax.get_legend_handles_labels()
    ev_h   = [mlines.Line2D([], [], color=c, linestyle=':', linewidth=1.8, label=lbl)
              for lbl, _, c in EVENTS]
    ax.legend(handles=data_h[0], labels=data_h[1], facecolor=BG_DARK,
              labelcolor=TEXT_PRIMARY, loc=loc)
    fig.legend(handles=ev_h, loc='lower center', bbox_to_anchor=(0.5, 0.0),
               ncol=min(len(EVENTS), 4), facecolor=BG_PANEL, edgecolor=TEXT_MUTED,
               labelcolor=TEXT_PRIMARY, fontsize=8.5, framealpha=0.95, borderpad=0.8)

print('Setup complete.')


## 1. Load data

In [ ]:
df = pd.read_parquet(DATA_PATH)

df['date'] = pd.to_datetime(df['created_utc'], utc=True).dt.normalize()
df['text_clean'] = df['text_clean'].fillna('')
df['text']       = df['text'].fillna('')

# Filter to project date range
df = df[(df['date'] >= '2024-07-05') & (df['date'] <= '2024-11-04')].reset_index(drop=True)

print(f'Posts     : {len(df):,}')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
print()
print(df['candidate'].value_counts())
print()
print(df['subreddit'].value_counts())


---
## 2. VADER

**What it is:** VADER (Valence Aware Dictionary and sEntiment Reasoner) is a rule-based lexicon
optimised for social media text. It returns a compound score ∈ [−1, +1].
Threshold: ≥ +0.05 → Positive, ≤ −0.05 → Negative, else Neutral.

In [ ]:
vader = SentimentIntensityAnalyzer()

def vader_scores(text):
    s = vader.polarity_scores(str(text))
    return s['compound'], s['pos'], s['neg'], s['neu']

df['vader_compound'], df['vader_pos'], df['vader_neg'], df['vader_neu'] =     zip(*df['text_clean'].map(vader_scores))

df['vader_label'] = pd.cut(df['vader_compound'],
                           bins=[-1.01, -0.05, 0.05, 1.01],
                           labels=['Negative', 'Neutral', 'Positive'])

print('VADER distribution:')
print(df['vader_label'].value_counts(normalize=True).mul(100).round(1).to_string())
print(f'\nMean compound: {df["vader_compound"].mean():+.4f}')


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
for cand, color in BUZZ_COLORS.items():
    sub = df[df['candidate'] == cand]['vader_compound']
    ax.hist(sub, bins=40, color=color, alpha=0.5, label=cand, edgecolor='none')
ax.axvline(0, color='white', linestyle='--', linewidth=1, alpha=0.6)
ax.set_title('VADER — Distribution of Compound Scores', color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
ax.set_xlabel('Compound score', color=TEXT_MUTED)
ax.set_ylabel('Count', color=TEXT_MUTED)
ax.tick_params(colors=TEXT_MUTED)
ax.legend(facecolor=BG_DARK, labelcolor=TEXT_PRIMARY)
plt.tight_layout(); plt.show()


In [ ]:
label_shares = (
    df.groupby(['candidate', 'vader_label']).size()
      .unstack(fill_value=0)
      .apply(lambda r: r / r.sum(), axis=1)
      .reindex(BUZZ_ORDER)[['Positive', 'Neutral', 'Negative']]
)
fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
x, bottom = np.arange(len(BUZZ_ORDER)), np.zeros(len(BUZZ_ORDER))
for label, color in POL_COLORS.items():
    vals = label_shares[label].values
    ax.bar(x, vals, bottom=bottom, color=color, label=label, edgecolor='none')
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 0.05:
            ax.text(x[i], b + v/2, f'{v:.0%}', ha='center', va='center', color='white', fontsize=9)
    bottom += vals
ax.set_xticks(x); ax.set_xticklabels(BUZZ_ORDER, color=TEXT_MUTED)
ax.set_ylabel('Share', color=TEXT_MUTED)
ax.set_title('VADER — Polarity Share per Buzz Group', color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
ax.tick_params(colors=TEXT_MUTED)
ax.legend(facecolor=BG_DARK, labelcolor=TEXT_PRIMARY)
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
for group, color in BUZZ_COLORS.items():
    daily = df[df['candidate'] == group].groupby('date')['vader_compound'].mean()
    roll  = daily.rolling(7, min_periods=1).mean()
    ax.plot(daily.index, daily.values, color=color, alpha=0.15, linewidth=1)
    ax.plot(roll.index, roll.values, color=color, linewidth=2, label=group)
ax.axhline(0, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
add_events(ax)
ax.set_title('VADER — Daily Compound Score per Buzz Group (7d rolling avg)',
             color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
ax.set_ylabel('VADER compound', color=TEXT_MUTED)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.tick_params(axis='x', rotation=30, colors=TEXT_MUTED)
ax.tick_params(axis='y', colors=TEXT_MUTED)
event_legend(fig, ax)
plt.tight_layout(rect=[0, 0.18, 1, 1]); plt.show()


---
## 3. Twitter-RoBERTa

**What it is:** `cardiffnlp/twitter-roberta-base-sentiment-latest` is a RoBERTa transformer
fine-tuned on ~124 million tweets. It understands context and informal language — well suited
for Reddit posts which share similar style with social media.

**Output:** Probabilities for Negative / Neutral / Positive.
We use `roberta_sentiment = prob_positive − prob_negative` ∈ [−1, +1].

**Applied to:** `text_clean` column.

In [ ]:
from transformers import pipeline

print('Loading cardiffnlp/twitter-roberta-base-sentiment-latest ...')
roberta = pipeline(
    'sentiment-analysis',
    model='cardiffnlp/twitter-roberta-base-sentiment-latest',
    tokenizer='cardiffnlp/twitter-roberta-base-sentiment-latest',
    max_length=512, truncation=True, device=-1
)
print('Model loaded.')


In [ ]:
BATCH_SIZE = 64
texts = df['text_clean'].fillna('').tolist()

pos_probs, neg_probs, neu_probs, labels = [], [], [], []

for i in range(0, len(texts), BATCH_SIZE):
    batch   = [str(t)[:512] for t in texts[i:i+BATCH_SIZE]]
    results = roberta(batch, top_k=None)
    for r in results:
        probs = {x['label'].lower(): x['score'] for x in r}
        pos = probs.get('positive', 0.0)
        neg = probs.get('negative', 0.0)
        neu = probs.get('neutral',  0.0)
        pos_probs.append(pos); neg_probs.append(neg); neu_probs.append(neu)
        labels.append(max(probs, key=probs.get).capitalize())
    if i % 5000 == 0:
        print(f'  {i:>6,} / {len(texts):,}', end='\r')

df['roberta_pos']       = pos_probs
df['roberta_neg']       = neg_probs
df['roberta_neu']       = neu_probs
df['roberta_sentiment'] = df['roberta_pos'] - df['roberta_neg']
df['roberta_label']     = labels

print(f'\nDone. {len(df):,} posts scored.')
print(df['roberta_label'].value_counts(normalize=True).mul(100).round(1).to_string())
print(f'\nMean roberta_sentiment: {df["roberta_sentiment"].mean():+.4f}')


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
ax.hist(df['roberta_sentiment'], bins=40, color=DEMOCRAT, alpha=0.85, edgecolor='none')
ax.axvline(0, color='white', linestyle='--', linewidth=1, alpha=0.6, label='Zero')
ax.axvline(df['roberta_sentiment'].mean(), color='yellow', linewidth=1.5,
           label=f'Mean: {df["roberta_sentiment"].mean():.4f}')
ax.set_title('Twitter-RoBERTa — Distribution of Sentiment Scores',
             color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
ax.set_xlabel('RoBERTa sentiment (pos − neg)', color=TEXT_MUTED)
ax.set_ylabel('Count', color=TEXT_MUTED)
ax.tick_params(colors=TEXT_MUTED)
ax.legend(facecolor=BG_DARK, labelcolor=TEXT_PRIMARY)
plt.tight_layout(); plt.show()


In [ ]:
label_shares_r = (
    df.groupby(['candidate', 'roberta_label']).size()
      .unstack(fill_value=0)
      .apply(lambda r: r / r.sum(), axis=1)
      .reindex(BUZZ_ORDER)[['Positive', 'Neutral', 'Negative']]
)
fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
x, bottom = np.arange(len(BUZZ_ORDER)), np.zeros(len(BUZZ_ORDER))
for label, color in POL_COLORS.items():
    vals = label_shares_r[label].values
    ax.bar(x, vals, bottom=bottom, color=color, label=label, edgecolor='none')
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 0.05:
            ax.text(x[i], b + v/2, f'{v:.0%}', ha='center', va='center', color='white', fontsize=9)
    bottom += vals
ax.set_xticks(x); ax.set_xticklabels(BUZZ_ORDER, color=TEXT_MUTED)
ax.set_ylabel('Share', color=TEXT_MUTED)
ax.set_title('Twitter-RoBERTa — Polarity Share per Buzz Group',
             color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
ax.tick_params(colors=TEXT_MUTED)
ax.legend(facecolor=BG_DARK, labelcolor=TEXT_PRIMARY)
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
for group, color in BUZZ_COLORS.items():
    daily = df[df['candidate'] == group].groupby('date')['roberta_sentiment'].mean()
    roll  = daily.rolling(7, min_periods=1).mean()
    ax.plot(daily.index, daily.values, color=color, alpha=0.15, linewidth=1)
    ax.plot(roll.index, roll.values, color=color, linewidth=2, label=group)
ax.axhline(0, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
add_events(ax)
ax.set_title('Twitter-RoBERTa — Daily Sentiment per Buzz Group (7d rolling avg)',
             color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
ax.set_ylabel('RoBERTa sentiment (pos − neg)', color=TEXT_MUTED)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.tick_params(axis='x', rotation=30, colors=TEXT_MUTED)
ax.tick_params(axis='y', colors=TEXT_MUTED)
event_legend(fig, ax)
plt.tight_layout(rect=[0, 0.18, 1, 1]); plt.show()


---
## 4. TextBlob

**What it is:** TextBlob uses the Pattern library lexicon to assign a polarity score ∈ [−1, +1]
and a subjectivity score ∈ [0, 1] (0 = very objective, 1 = very subjective).

**Applied to:** `text_clean` column.

In [ ]:
def tb_scores(text):
    b = TextBlob(str(text))
    return b.sentiment.polarity, b.sentiment.subjectivity

tb_res = df['text_clean'].apply(tb_scores)
df['tb_polarity']     = [r[0] for r in tb_res]
df['tb_subjectivity'] = [r[1] for r in tb_res]
df['tb_label'] = pd.cut(df['tb_polarity'],
                        bins=[-1.01, -0.05, 0.05, 1.01],
                        labels=['Negative', 'Neutral', 'Positive'])

print('TextBlob distribution:')
print(df['tb_label'].value_counts(normalize=True).mul(100).round(1).to_string())
print(f'\nMean polarity     : {df["tb_polarity"].mean():+.4f}')
print(f'Mean subjectivity : {df["tb_subjectivity"].mean():.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor(BG_DARK)
for ax in axes: ax.set_facecolor(BG_PANEL)

axes[0].hist(df['tb_polarity'], bins=40, color=ACCENT, alpha=0.85, edgecolor='none')
axes[0].axvline(0, color='white', linestyle='--', linewidth=1, alpha=0.6)
axes[0].set_title('TextBlob — Polarity', color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
axes[0].set_xlabel('Polarity', color=TEXT_MUTED); axes[0].tick_params(colors=TEXT_MUTED)

axes[1].hist(df['tb_subjectivity'], bins=40, color=PURPLE, alpha=0.85, edgecolor='none')
axes[1].set_title('TextBlob — Subjectivity', color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
axes[1].set_xlabel('Subjectivity', color=TEXT_MUTED); axes[1].tick_params(colors=TEXT_MUTED)

plt.tight_layout(); plt.show()


In [ ]:
label_shares_tb = (
    df.groupby(['candidate', 'tb_label']).size()
      .unstack(fill_value=0)
      .apply(lambda r: r / r.sum(), axis=1)
      .reindex(BUZZ_ORDER)[['Positive', 'Neutral', 'Negative']]
)
fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
x, bottom = np.arange(len(BUZZ_ORDER)), np.zeros(len(BUZZ_ORDER))
for label, color in POL_COLORS.items():
    vals = label_shares_tb[label].values
    ax.bar(x, vals, bottom=bottom, color=color, label=label, edgecolor='none')
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 0.05:
            ax.text(x[i], b + v/2, f'{v:.0%}', ha='center', va='center', color='white', fontsize=9)
    bottom += vals
ax.set_xticks(x); ax.set_xticklabels(BUZZ_ORDER, color=TEXT_MUTED)
ax.set_ylabel('Share', color=TEXT_MUTED)
ax.set_title('TextBlob — Polarity Share per Buzz Group',
             color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
ax.tick_params(colors=TEXT_MUTED)
ax.legend(facecolor=BG_DARK, labelcolor=TEXT_PRIMARY)
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
for group, color in BUZZ_COLORS.items():
    daily = df[df['candidate'] == group].groupby('date')['tb_polarity'].mean()
    roll  = daily.rolling(7, min_periods=1).mean()
    ax.plot(daily.index, daily.values, color=color, alpha=0.15, linewidth=1)
    ax.plot(roll.index, roll.values, color=color, linewidth=2, label=group)
ax.axhline(0, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
add_events(ax)
ax.set_title('TextBlob — Daily Polarity per Buzz Group (7d rolling avg)',
             color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
ax.set_ylabel('TextBlob polarity', color=TEXT_MUTED)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.tick_params(axis='x', rotation=30, colors=TEXT_MUTED)
ax.tick_params(axis='y', colors=TEXT_MUTED)
event_legend(fig, ax)
plt.tight_layout(rect=[0, 0.18, 1, 1]); plt.show()


---
## 5. NRCLex — Emotion Detection

**What it is:** NRCLex maps words to 8 primary emotions (Plutchik's wheel): fear, anger,
anticipation, trust, surprise, sadness, disgust, joy. Each score is the fraction of
emotionally-tagged words matching that emotion.

**Applied to:** `text_clean` column.

In [ ]:
EMOTION_COLORS = {
    'fear': REPUBLICAN, 'anger': '#e67e22', 'anticipation': ACCENT,
    'trust': '#2ecc71', 'surprise': NEUTRAL, 'sadness': DEMOCRAT,
    'disgust': '#9b59b6', 'joy': '#FFD700',
}

def nrc_scores(text):
    af = NRCLex(str(text)).affect_frequencies
    return {e: af.get(e, 0) for e in EMOTION_COLS + ['positive', 'negative']}

print('Running NRCLex ...')
nrc_df = df['text_clean'].apply(nrc_scores).apply(pd.Series)
df     = pd.concat([df, nrc_df], axis=1)
df['nrc_sentiment'] = df['positive'] - df['negative']
print('Done.')


In [ ]:
avg = df[EMOTION_COLS].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
colors = [EMOTION_COLORS[e] for e in avg.index]
ax.bar(avg.index, avg.values, color=colors, edgecolor='none')
ax.set_title('NRCLex — Mean Emotion Profile (all posts)',
             color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
ax.set_ylabel('Mean frequency', color=TEXT_MUTED)
ax.tick_params(colors=TEXT_MUTED)
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
fig.patch.set_facecolor(BG_DARK)
for ax, group in zip(axes, BUZZ_ORDER):
    sub = df[df['candidate'] == group][EMOTION_COLS].mean().sort_values(ascending=False)
    colors = [EMOTION_COLORS[e] for e in sub.index]
    ax.set_facecolor(BG_PANEL)
    ax.bar(sub.index, sub.values, color=colors, edgecolor='none')
    ax.set_title(group, color=TEXT_PRIMARY, fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, colors=TEXT_MUTED)
    ax.tick_params(axis='y', colors=TEXT_MUTED)
axes[0].set_ylabel('Mean frequency', color=TEXT_MUTED)
fig.suptitle('NRCLex — Emotion Profile per Buzz Group',
             color=TEXT_PRIMARY, fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
KEY_EMOTIONS = {'trust': '#2ecc71', 'anticipation': ACCENT, 'fear': REPUBLICAN}
fig, ax = plt.subplots(figsize=(13, 4))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
for emotion, color in KEY_EMOTIONS.items():
    daily = df.groupby('date')[emotion].mean()
    roll  = daily.rolling(7, min_periods=1).mean()
    ax.plot(daily.index, daily.values, color=color, alpha=0.15, linewidth=1)
    ax.plot(roll.index, roll.values, color=color, linewidth=2, label=emotion)
add_events(ax)
ax.set_title('NRCLex — Trust / Anticipation / Fear over Time (7d rolling avg)',
             color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
ax.set_ylabel('Mean emotion frequency', color=TEXT_MUTED)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.tick_params(axis='x', rotation=30, colors=TEXT_MUTED)
ax.tick_params(axis='y', colors=TEXT_MUTED)
event_legend(fig, ax)
plt.tight_layout(rect=[0, 0.18, 1, 1]); plt.show()


---
## 6. Method Comparison

Correlation between the four sentiment scores and a z-scored overlay time series.

In [ ]:
sent_cols = {
    'VADER':    'vader_compound',
    'RoBERTa':  'roberta_sentiment',
    'TextBlob': 'tb_polarity',
    'NRCLex':   'nrc_sentiment',
}
corr = (
    df[[v for v in sent_cols.values()]]
    .rename(columns={v: k for k, v in sent_cols.items()})
    .corr()
)
print('Correlation matrix:')
print(corr.round(3))


In [ ]:
METHOD_COLORS = {'VADER': ACCENT, 'RoBERTa': DEMOCRAT, 'TextBlob': PURPLE, 'NRCLex': '#FFD700'}
fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
for label, col in sent_cols.items():
    daily = df.groupby('date')[col].mean()
    roll  = daily.rolling(7, min_periods=1).mean()
    z     = (roll - roll.mean()) / (roll.std() + 1e-8)
    ax.plot(z.index, z.values, linewidth=2, label=label, color=METHOD_COLORS[label])
ax.axhline(0, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
add_events(ax)
ax.set_title('All Methods — Daily Sentiment (7d rolling avg, z-scored)',
             color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
ax.set_ylabel('Z-scored sentiment', color=TEXT_MUTED)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.tick_params(axis='x', rotation=30, colors=TEXT_MUTED)
ax.tick_params(axis='y', colors=TEXT_MUTED)
event_legend(fig, ax)
plt.tight_layout(rect=[0, 0.18, 1, 1]); plt.show()


---
## 7. Save enriched data

In [ ]:
save_cols = [
    'date', 'candidate', 'subreddit', 'text', 'text_clean',
    'roberta_sentiment', 'roberta_pos', 'roberta_neg', 'roberta_neu', 'roberta_label',
    'nrc_sentiment',
] + EMOTION_COLS

df[[c for c in save_cols if c in df.columns]].to_csv(OUT_PATH, index=False)
print(f'Saved {len(df):,} rows → {OUT_PATH}')
